# Popcorn + `torch.compile`

Consider the same model handed to `torch.compile` instead of having its kernels swapped by hand. Inductor fuses a good deal on its own, but it will not reach for a hand-written kernel it does not know about, so the ops that need one are left as they are.

In this example, we let Popcorn add its patterns to Inductor, so the compiled graph picks those kernels up without the model changing at all. This is experimental, and here we only rewrite the language-model loss.

In [1]:
import torch

import popcorn.compile
from modeling import CausalLM, ModelConfig
from popcorn.bench import compare, report

torch.manual_seed(0)
device = "cuda"
config = ModelConfig(layer_types=("attn",), vocab_size=131072)
model = CausalLM(config).to(device)
batch = {
    "input_ids": torch.randint(config.vocab_size, (16, 1024), device=device),
    "labels": torch.randint(config.vocab_size, (16, 1024), device=device),
}

Start with plain `torch.compile` and nothing else. The measured step includes the backward pass and the gradient cleanup, so what we are timing is a whole training step rather than a forward.

In [2]:
def train_step(**batch):
    loss = model(**batch)
    loss.backward()
    model.zero_grad(set_to_none=True)
    return loss.detach()


compiled = torch.compile(train_step)
plain = compare(compiled, train_step, batch, backward=False, repeats=3)
report(plain, "torch.compile", mine="compiled", reference="eager");

torch.compile eager    981.0 ms -> compiled  939.9 ms   (1.0x)
              peak        36.13 GB ->     18.63 GB
              err 0.00e+00 on scale 12.0


Such a step spends much of its memory on one tensor: the full logits for a 131K vocabulary, written out in full before the loss reduces them to a scalar. Popcorn has a fused loss that never materializes them, and `enable()` registers it as an Inductor pattern.

The model is untouched, so only the compiled function has to change. Reset once so it recompiles with the new pattern.

In [3]:
popcorn.compile.enable(ops=["linear_cross_entropy"])
torch._dynamo.reset()

fused = compare(compiled, train_step, batch, backward=False, repeats=3)
report(fused, "+ popcorn", mine="compiled", reference="eager")
saved = (plain.bench["fwd_mem_mb"] - fused.bench["fwd_mem_mb"]) / 1024
print(f"{saved:.1f} GB less than plain compile")

+ popcorn eager    982.0 ms -> compiled  949.6 ms   (1.0x)
          peak        36.13 GB ->     12.76 GB
          err 0.00e+00 on scale 12.0
5.9 GB less than plain compile


As can be seen, the win here is memory rather than time. The step takes about as long either way, but the logits are gone from the peak, and that headroom is what lets you raise the batch size or the sequence length.

`enable()` only rewrites the ops you name, so everything else compiles exactly as it did before. It stays experimental, which is why you have to ask for it.